# DatasetManager 完整生命周期演示

本 Notebook 使用真实 PostgreSQL/pgvector、PyIceberg 与 MinIO。所有图片已经提交在 `materials/raw_images`，运行时不依赖源 sample_1000。

In [ ]:
from pathlib import Path
import pandas as pd
from image_gallery.importers import LocalPathParser
from examples.dataset_manager_demo.helpers import (
    DemoDatasetImporter, load_demo_config, probe_existing_backend, require_ready_backend,
    start_demo_backend, stop_demo_backend, open_demo_clients, validate_sample_manifest,
)
DEMO_ROOT = Path.cwd() / 'examples' / 'dataset_manager_demo'
if not DEMO_ROOT.exists():
    DEMO_ROOT = Path.cwd()
MATERIALS = DEMO_ROOT / 'materials'
validate_sample_manifest(MATERIALS / 'sample_manifest.json')


## 1. 环境探测

优先只读探测仓库 `.env`。配置缺失或连接失败时不会自动创建、删除或重建服务。请参考 `.env.example` 修改配置。

In [ ]:
env_candidates = [DEMO_ROOT / '.env', DEMO_ROOT.parent.parent / '.env']
env_path = next((path for path in env_candidates if path.exists()), env_candidates[0])
probe = probe_existing_backend(env_path)
print('已有环境可用' if probe.ready else '\n'.join(probe.diagnostics))


### 可选：显式创建或重新创建演示 Backend

若没有可用 `.env`，取消下一单元格注释。该命令只创建本次 Notebook 持有的临时服务；不会改写 `.env`。

In [ ]:
demo_backend = None
# demo_backend = start_demo_backend(demo_root=DEMO_ROOT / 'runtime', recreate=True)
config = demo_backend.config if demo_backend is not None else require_ready_backend(probe)
storage, manager, prefix_id = open_demo_clients(config)


## 2. 导入与 V1

创建 Repo、Dataset、Storage Prefix 绑定和 Tag，再将 20 张本地图片托管写入 MinIO，并作为一次原子 commit 发布。

In [ ]:
repo = manager.create_repo(name='Dataset Demo')
repo.bind_storage_prefix(prefix_id=prefix_id)
dataset = repo.create_dataset(name='Images')
raw_tag = repo.create_tag(name='raw')
v1 = DemoDatasetImporter(
    source=LocalPathParser(MATERIALS / 'raw_images'), dataset=dataset, base=dataset.open_branch(),
    storage_manager=storage, prefix_id=prefix_id, tag_ids=[raw_tag.tag_id],
).run()
assert v1.imported_count == 20
print('V1 图片数:', v1.view.count(), '首张 bytes:', len(v1.view.read_image(asset_id=v1.asset_ids[0])))


## 3. 分支与版本

Checkpoint 和旧 View 都固定在 V1；`main` 与 `experiment` 从相同状态开始后独立推进。

In [ ]:
checkpoint = dataset.create_checkpoint(name='v1', source=v1.view)
dataset.create_branch(name='experiment', source=checkpoint)
main_row = v1.view.get_row(asset_id=v1.asset_ids[0]).to_dict(); main_row['source_uri'] = 'demo://main-v2'
main_v2 = dataset.commit(branch='main', base=dataset.open_branch(), frame=pd.DataFrame([main_row]), mode='upsert').view
experiment_row = v1.view.get_row(asset_id=v1.asset_ids[1]).to_dict(); experiment_row['source_uri'] = 'demo://experiment-v2'
experiment_v2 = dataset.commit(branch='experiment', base=dataset.open_branch(name='experiment'), frame=pd.DataFrame([experiment_row]), mode='upsert').view
assert checkpoint.scan().equals(v1.view.scan())
print('main snapshot:', main_v2.snapshot_id, 'experiment snapshot:', experiment_v2.snapshot_id)


## 4. 模型托管的向量生成

DatasetManager 的 ModelManager 会持久化冻结模型定义。VectorField 只保存 `model_id`、配置指纹、维度和 dtype 的强绑定，调用方不能直接提交向量。

In [ ]:
model = manager.model_manager.get(model_id='demo-image-length-v1')
assert model is not None
field = repo.schema.add_vector(name='demo-vector', model_id=model.model_id, distance='cosine')
assert field.model_fingerprint == model.fingerprint
assert field.dimension == model.dimension and field.numeric_type == model.dtype
print('模型:', model.model_id, '维度:', field.dimension, '指纹:', field.model_fingerprint[:20] + '...')


### 默认 Head 生成与统一读取

不传 `source` 时，`generate_embed()` 固定 `main` 当前 Head。向量发布在 Repo 当前空间，不会创建或移动 Iceberg Snapshot；扫描时普通列和向量列仍通过同一个 `fields` 参数返回 DataFrame。

In [ ]:
snapshot_before_embed = dataset.open_branch().snapshot_id
embed_result = dataset.generate_embed(field='demo-vector')
assert embed_result.source_snapshot_id == main_v2.snapshot_id
assert embed_result.generated == 20 and embed_result.updated == 0 and embed_result.skipped == 0
assert dataset.open_branch().snapshot_id == snapshot_before_embed
vector_frame = main_v2.scan(fields=['asset_id', 'source_uri', 'demo-vector'])
assert isinstance(vector_frame, pd.DataFrame)
assert vector_frame['demo-vector'].notna().all()
vector_frame.head(3)


### 增量跳过、覆盖与指定 View

默认 `overwrite=False` 会跳过已有值；`overwrite=True` 会重新生成并替换当前 View 的向量。显式传入固定 View 时只扫描该 View 的成员。由于 Repo 当前向量按 `asset_id` 复用，experiment 与 main 共享的图片会直接跳过。

In [ ]:
skipped_result = dataset.generate_embed(field='demo-vector')
assert skipped_result.generated == 0 and skipped_result.updated == 0 and skipped_result.skipped == 20
overwrite_result = dataset.generate_embed(field='demo-vector', overwrite=True)
assert overwrite_result.generated == 0 and overwrite_result.updated == 20 and overwrite_result.skipped == 0
experiment_result = dataset.generate_embed(field='demo-vector', source=experiment_v2)
assert experiment_result.source_snapshot_id == experiment_v2.snapshot_id
assert experiment_result.generated == 0 and experiment_result.updated == 0 and experiment_result.skipped == 20
assert dataset.open_branch().snapshot_id == snapshot_before_embed
print('跳过:', skipped_result, '覆盖:', overwrite_result, '指定 View:', experiment_result)


## 5. 回退与 Clone

Clone 从固定 View 建立独立历史；main 回退到祖先 Checkpoint 不影响 experiment。Repo 当前向量也不会随 Dataset 历史回退。

In [ ]:
clone = repo.clone_dataset(source=main_v2, name='Clone')
rolled_back = dataset.rollback(branch='main', base=main_v2, checkpoint=checkpoint)
assert rolled_back.scan().equals(checkpoint.scan())
assert dataset.open_branch(name='experiment').snapshot_id == experiment_v2.snapshot_id
assert clone.open_branch().snapshot_id is not None


## 6. 关闭并重新连接

关闭第一组客户端后用相同配置重建。稳定 Prefix、冻结模型定义、VectorField 和 Repo 当前向量都会从持久层恢复。

In [ ]:
manager.close(); storage.close()
storage, manager, prefix_id = open_demo_clients(config)
reopened_repo = manager.open_repo(name='dataset demo')
reopened_dataset = reopened_repo.open_dataset(name='images')
reopened_model = manager.model_manager.get(model_id='demo-image-length-v1')
reopened_field = reopened_repo.schema.get_vector(name='demo-vector')
assert reopened_model is not None and reopened_model.fingerprint == field.model_fingerprint
assert reopened_field.model_id == reopened_model.model_id
assert reopened_dataset.open_checkpoint(name='v1').count() == 20
assert reopened_repo.open_dataset(name='clone').open_branch().count() == 20
reopened_vector_frame = reopened_dataset.open_branch().scan(fields=['asset_id', 'demo-vector'])
assert reopened_vector_frame['demo-vector'].notna().all()
reopened_vector_frame.head(3)


## 7. 清理

以下命令会关闭客户端。只有本 Notebook 显式创建的 managed session 才允许停止容器；existing 环境不会被停止或删除。

In [ ]:
manager.close(); storage.close()
# if demo_backend is not None:
#     stop_demo_backend(demo_backend, remove_volumes=True)
